In [ ]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.constants import START,END
from langgraph.graph import StateGraph


#State的数据处理方法:Reducer数据归约器

class CustomReducerState(TypedDict):
    count:int  #整个数据线么有指定Reducer，将采用默认行为:新值覆盖旧值
    nodes:Annotated[ list[str],add ] #给数据项指定了规约器，行为是:旧值 + 新值

#定义节点
def node1(state:CustomReducerState):
    print(f"node1节点得到数据：{state}")
    return {'count':1,"nodes":["node1"]}  #注意:nodes的类型声明为list[str]，更新数据也得是list[str]

def node2(state:CustomReducerState):
    print(f"node2节点收到数据:{state}")
    return  {'count':2,'nodes':['node2']}

#构建图
builder = StateGraph(CustomReducerState)
builder.add_node('node1',node1)
builder.add_node('node2',node2)

builder.add_edge(START,'node1')
builder.add_edge('node1','node2')
builder.add_edge('node2',END)

graph = builder.compile()

#调用图
res = graph.invoke(input={'count':0,'nodes':[]})
print(res)

In [ ]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.constants import START,END
from langgraph.graph import StateGraph

class NormalState(TypedDict):
    nodes:Annotated[ list[str],add ]

def node1(state:NormalState):
    print(f"node1节点收到：{state}")
    return {'nodes':['node1']}
def node21(state:NormalState):
    print(f"node21节点收到：{state}")
    return {'nodes':['node21']}
def node22(state:NormalState):
    print(f"node22节点收到：{state}")
    return {'nodes':['node22']}
def node3(state:NormalState):
    print(f"node3节点收到：{state}")
    return {'nodes':['node3']}

builder = StateGraph(NormalState)
builder.add_node("node1",node1)
builder.add_node("node21",node21)
builder.add_node("node22",node22)
builder.add_node("node3",node3)

builder.add_edge(START,'node1')
builder.add_edge('node1','node21')
builder.add_edge('node1','node22')
builder.add_edge('node21','node3')
builder.add_edge('node22','node3')
builder.add_edge('node3',END)

graph = builder.compile()

res = graph.invoke(input = {'nodes':[]})
print(res)

In [2]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.constants import START,END
from langgraph.graph import StateGraph
#条件便的实现方式一:add_conditional_edge(初始节点,节点函数)
class ScoreState(TypedDict):
    score:float
    result:str

def input_score(state:ScoreState):
    score = float(input('请输入成绩:'))
    return {'score':score}
def pass_node(stade:ScoreState):
    return {'result':'通过'}
def fail_node(stade:ScoreState):
    return {'result':'不通过'}

#为条件边准备的路由函数：用于判断下个节点的名称
def score_router(state:ScoreState):
    if state['score'] >= 60.0:
        return  'pass'
    else:
        return 'fail'

builder = StateGraph(ScoreState)
builder.add_node('score',input_score)
builder.add_node('pass',pass_node)
builder.add_node('fail',fail_node)

builder.add_edge(START,'score')
builder.add_conditional_edges('score',score_router)
builder.add_edge('pass',END)
builder.add_edge('fail',END)
graph = builder.compile()

res = graph.invoke(input = {})
print(res)

{'score': 50.2, 'result': '不通过'}


In [18]:
from langgraph.types import Command
#条件边的另一种实现:不需要add_conditional_edge,在节点函数用return command
class CommandState(TypedDict):
    score:float
    result:str

def score_input(state:CommandState):
    score = float(input("请输入分数"))
    return Command(
        update = {'score':score},
        goto = 'pass' if score>= 60 else 'fail'
    )

def pass_node(state:CommandState):
    return {'result':"通过"}

def fail_node(state:CommandState):
    return  {'result':"不通过"}

builder = StateGraph(CommandState)
builder.add_node('score',score_input)
builder.add_node('pass',pass_node)
builder.add_node('fail',fail_node)

builder.add_edge(START,'score')
builder.add_edge('pass',END)
builder.add_edge('fail',END)
graph = builder.compile()
res = graph.invoke(input={})
print(res)


{'score': 59.99999999, 'result': '不通过'}
